# 00 · Construcción del Catálogo de Cúmulos
**Metodología:** §2.1 de Mayes+2026

A partir del catálogo de group IDs (generado por tu código de estado dinámico),
cargamos las propiedades necesarias de TNG-100 usando `illustris_python`
y construimos el catálogo base para el análisis ICL/BCG.

**Campos que se extraen:**
- M₂₀₀c, R₂₀₀c, posición del halo
- ID del BCG (GroupFirstSub)
- Tiempo desde el último merger mayor del BCG (árbol SubLink)

**Lo que NO se hace aquí:** estado dinámico (lo calcula tu código existente).
El estado dinámico puede añadirse al catálogo como columna adicional.


In [1]:
import sys, os, pickle
import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM

# Agregar el código original al path (solo lectura, no se modifica)
sys.path.insert(0, './original_shift_code')
import illustris_python as il
import params_icl as P          # copia local de params para este análisis

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 12,
                     'axes.spines.top': False, 'axes.spines.right': False})

cosmo = FlatLambdaCDM(H0=67.74, Om0=0.3089)
print(f"basePath : {P.basePath}")
print(f"Snapshot : {P.SNAP}")
print(f"h = {P.h},  UL = {P.UL:.4f},  UM = {P.UM:.3e}")


basePath : /home/tnguser/sims.TNG/TNG100-1/output/
Snapshot : 99
h = 0.6774,  UL = 1.4762,  UM = 1.476e+10


## 1 · Leer el catálogo de group IDs

In [2]:
# ── Cargar el pickle con los grupos a analizar ────────────────────────────
with open(P.CATALOG_PKL, 'rb') as f:
    cat_raw = pickle.load(f)

# Inspeccionar la estructura
print("Tipo:", type(cat_raw))
if isinstance(cat_raw, dict):
    print("Claves:", list(cat_raw.keys()))
    # Muestra las primeras entradas de cada campo
    for k, v in cat_raw.items():
        try: print(f"  {k}: shape={np.shape(v)}, primeros valores={np.asarray(v).flat[:3]}")
        except: print(f"  {k}: {v}")
else:
    print("Shape:", np.shape(cat_raw))
    print("Primeros valores:", np.asarray(cat_raw)[:5])

Tipo: <class 'dict'>
Claves: ['05FossilR200', '05M200_F_R200', '05R200_F_R200', '05Deltamag_F', '05NFossilR200', '05M200_NF_R200', '05R200_NF_R200', '05Deltamag_NF', '1FossilR200', '1M200_F_R200', '1R200_F_R200', '1Deltamag_F', '1NFossilR200', '1M200_NF_R200', '1R200_NF_R200', '1Deltamag_NF', '05FossilISO', '05M200_F_ISO', '05R200_F_ISO', '1FossilISO', '1M200_F_ISO', '1R200_F_ISO']
  05FossilR200: shape=(55,), primeros valores=[0 1 2]
  05M200_F_R200: shape=(55,), primeros valores=[14.571739 14.575134 14.528206]
  05R200_F_R200: shape=(55,), primeros valores=[1517.8958 1521.8263 1467.9613]
  05Deltamag_F: shape=(55, 1), primeros valores=[2.6165848 2.1808262 2.4461632]
  05NFossilR200: shape=(65,), primeros valores=[ 4  5 11]
  05M200_NF_R200: shape=(65,), primeros valores=[14.4044    14.293855  14.1173115]
  05R200_NF_R200: shape=(65,), primeros valores=[1334.899  1226.3127 1070.9315]
  05Deltamag_NF: shape=(65, 1), primeros valores=[1.9999638 1.2905025 1.7921524]
  1FossilR200: shape=

In [ ]:
# ── Extraer índices de grupo del catálogo fossil/no-fossil ────────────────
#
# El pkl es un dict de clasificación fossil generado por tu código.
# Las claves contienen los índices TNG directamente (enteros 0-based):
#
#   '05FossilR200'  / '05NFossilR200'  → criterio 0.5*R200
#   '1FossilR200'   / '1NFossilR200'   → criterio 1.0*R200
#   '05FossilISO'   / '1FossilISO'     → criterio de aislamiento
#
# Combinamos fossil + no-fossil para el análisis ICL (todos los cúmulos).
# Cambia las claves si quieres analizar solo un subconjunto.

fossil_idx    = np.array(cat_raw['05FossilR200'],  dtype=int)
nonfossil_idx = np.array(cat_raw['05NFossilR200'], dtype=int)
group_idx     = np.unique(np.concatenate([fossil_idx, nonfossil_idx]))

n_cl = len(group_idx)
print(f"Fossil    (0.5*R200) : {len(fossil_idx)} cúmulos")
print(f"No-fossil (0.5*R200) : {len(nonfossil_idx)} cúmulos")
print(f"Total a analizar     : {n_cl} cúmulos (únicos)")
print(f"Rango de índices     : {group_idx.min()} – {group_idx.max()}")

## 2 · Cargar propiedades del catálogo de halos TNG

In [ ]:
# Cargar header y catálogos completos de halos y subhalos
Header = il.groupcat.loadHeader(P.basePath, P.SNAP)
print("BoxSize (ckpc/h):", Header['BoxSize'])
print("Redshift        :", Header['Redshift'])
print("Time (a)        :", Header['Time'])

# Catálogo de halos (todos los grupos de TNG100-1)
Halos_all = il.groupcat.loadHalos(P.basePath, P.SNAP, fields=P.HALOS_FIELDS)
print(f"\nTotal de halos en TNG100-1 snap {P.SNAP}: {Halos_all['count']}")


In [ ]:
# Extraer sólo los halos que están en nuestro catálogo
M200c_raw = Halos_all['Group_M_Crit200'][group_idx]   # 1e10 M☉/h
R200c_raw = Halos_all['Group_R_Crit200'][group_idx]   # ckpc/h
GroupPos  = Halos_all['GroupPos'][group_idx] * P.UL   # kpc físicos
GroupCM   = Halos_all['GroupCM'][group_idx]  * P.UL   # kpc físicos
first_sub = Halos_all['GroupFirstSub'][group_idx]     # índice del BCG

# Convertir a unidades físicas
M200c = M200c_raw * P.UM     # M☉
R200c = R200c_raw * P.UL     # kpc físicos

print(f"N cúmulos         : {n_cl}")
print(f"log M200c (M☉)    : {np.log10(M200c.min()):.2f} – {np.log10(M200c.max()):.2f}")
print(f"R200c [kpc]       : {R200c.min():.0f} – {R200c.max():.0f}")
print(f"Primeros GroupFirstSub: {first_sub[:5]}")


## 3 · Posición del BCG para cada cúmulo

In [ ]:
# Cargar posiciones de subhalos para identificar el BCG
Subhalos_all = il.groupcat.loadSubhalos(P.basePath, P.SNAP, fields=P.SUBHALOS_FIELDS)

bcg_pos = Subhalos_all['SubhaloPos'][first_sub] * P.UL   # kpc físicos

# SubhaloMassType puede volver aplanado (N*6,) según la versión de illustris_python
raw_mtype = Subhalos_all['SubhaloMassType']
mtype = raw_mtype.reshape(-1, 6) if raw_mtype.ndim == 1 else raw_mtype
bcg_M_star = mtype[first_sub, 4] * P.UM   # masa estelar BCG [M☉]

print("Posición BCG (primeros 3):")
for i in range(min(3, n_cl)):
    print(f"  Cúmulo {group_idx[i]}: pos = {bcg_pos[i]}")
print(f"\nMasa estelar BCG: {np.log10(bcg_M_star.min()):.2f} – {np.log10(bcg_M_star.max()):.2f} log M☉")

## 4 · Tiempo desde el último merger mayor del BCG

Usamos el árbol SubLink de TNG para trazar el Main Progenitor Branch (MPB) del BCG
y encontrar el último evento de fusión con razón de masa estelar ≥ 1/5.

Referencia: Mayes+2026 §2 + Rodríguez-Gómez+2015


In [ ]:
def time_since_last_major_merger(bcg_id, basePath, snap, cosmo,
                                    mass_ratio_min=0.2):
    """
    Traza el árbol MPB del BCG y encuentra el último merger mayor.

    Parámetros
    ----------
    bcg_id        : int, índice del subhalo BCG en snap
    mass_ratio_min: float, razón mínima de masa estelar (defecto 1/5)

    Devuelve
    --------
    t_lookback : float, tiempo de lookback en Gyr desde z=0 hasta el merger.
                 NaN si no hay merger mayor registrado.
    """
    fields = ['SnapNum', 'SubhaloMassType', 'SubhaloID',
              'NextProgenitorID', 'FirstProgenitorID', 'MainLeafProgenitorID']
    try:
        tree = il.sublink.loadTree(basePath, snap, bcg_id, fields=fields,
                                    onlyMPB=False)
    except Exception as e:
        return np.nan

    if tree is None or 'SnapNum' not in tree:
        return np.nan

    snap_arr   = tree['SnapNum']
    mass_arr   = tree['SubhaloMassType'][:, 4]   # masa estelar (tipo 4)

    # Masa del progenitor principal en cada snapshot
    # Los snapshots están ordenados de z=0 hacia atrás en el MPB
    # Para detectar mergers: buscar el snap donde la masa del 2do progenitor
    # es significativa respecto a la del progenitor principal.
    # Usando la masa del PROG2: M_main[snap] - M_main[snap+1] (masa que se sumó)
    # Aproximación: Δm = masa sumada en cada paso del árbol
    # La masa del 2do progenitor = masa del subtree que se une = Δm

    # Identificamos mergers como: M_main(t-1) salta más que factor respecto a M_main(t)
    # (el MPB tiene masa creciente hacia z=0, decrece hacia atrás)
    # Un merger mayor ocurre cuando la masa cae significativamente hacia atrás:
    # ratio = M_main[i+1] / M_main[i] < (1 - mass_ratio_min / (1 + mass_ratio_min))
    t_last = np.nan
    for i in range(len(snap_arr) - 1):
        m_now  = mass_arr[i]
        m_prev = mass_arr[i + 1]
        if m_now <= 0 or m_prev <= 0:
            continue
        # Masa del 2do progenitor ≈ masa ganada en este paso
        m2 = m_now - m_prev
        if m2 <= 0:
            continue
        ratio = m2 / m_prev   # ≈ M2/M1 en el momento del merger
        if ratio >= mass_ratio_min:
            snap_merger = snap_arr[i]
            z_merger    = il.groupcat.loadHeader(basePath, int(snap_merger))['Redshift']
            t_last      = cosmo.lookback_time(z_merger).value   # Gyr
            break   # primer (más reciente) merger mayor encontrado

    return t_last

print("Calculando tiempo desde el último merger mayor...")
t_last_merger = np.full(n_cl, np.nan)
for i, sub_id in enumerate(first_sub):
    t_last_merger[i] = time_since_last_major_merger(
        int(sub_id), P.basePath, P.SNAP, cosmo)
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{n_cl}  t_med = {np.nanmedian(t_last_merger[:i+1]):.2f} Gyr", end="\r")
print(f"\nMerger mayor encontrado en {np.isfinite(t_last_merger).sum()}/{n_cl} cúmulos")


## 5 · Guardar el catálogo

In [ ]:
# Guardar como HDF5 para uso en los siguientes notebooks
with h5py.File(P.CATALOG_OUT, 'w') as f:
    f.attrs['basePath']    = P.basePath
    f.attrs['snap']        = P.SNAP
    f.attrs['n_clusters']  = n_cl

    f.create_dataset('group_idx',     data=group_idx)    # índice FoF en TNG
    f.create_dataset('M200c_Msun',    data=M200c)        # M☉
    f.create_dataset('R200c_kpc',     data=R200c)        # kpc físicos
    f.create_dataset('GroupPos_kpc',  data=GroupPos)     # kpc físicos (3D)
    f.create_dataset('GroupCM_kpc',   data=GroupCM)      # kpc físicos (3D)
    f.create_dataset('bcg_sub_idx',   data=first_sub)    # índice subhalo BCG
    f.create_dataset('bcg_pos_kpc',   data=bcg_pos)      # kpc físicos (3D)
    f.create_dataset('bcg_Mstar_Msun',data=bcg_M_star)   # M☉
    f.create_dataset('t_last_merger_Gyr', data=t_last_merger)  # Gyr

print(f"Catálogo guardado en: {P.CATALOG_OUT}")
print(f"  N cúmulos          : {n_cl}")
print(f"  log M200c [M☉]     : {np.log10(M200c.min()):.2f} – {np.log10(M200c.max()):.2f}")


## 6 · Visualización rápida del catálogo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(np.log10(M200c), bins=20, color='steelblue', edgecolor='white', lw=0.5)
ax.axvline(np.log10(M200c.min()), ls='--', color='k', lw=1.2, label='Límite mínimo')
ax.set_xlabel(r'$\log_{10}(M_{200c}\,/\,M_\odot)$')
ax.set_ylabel('N cúmulos')
ax.set_title('Distribución de masas')
ax.legend(fontsize=9)

ax = axes[1]
valid = np.isfinite(t_last_merger)
ax.scatter(np.log10(M200c[valid]), t_last_merger[valid],
           s=20, alpha=0.8, color='tomato')
ax.set_xlabel(r'$\log_{10}(M_{200c}\,/\,M_\odot)$')
ax.set_ylabel('Lookback time último merger mayor [Gyr]')
ax.set_title('Tiempo desde merger (Fig. 5 input)')

plt.tight_layout()
plt.savefig('fig_catalogo.pdf', bbox_inches='tight')
plt.show()

# Añadir estado dinámico al catálogo si ya lo tienes calculado:
# with h5py.File(P.CATALOG_OUT, 'a') as f:
#     f.create_dataset('dyn_state', data=mi_array_dyn_state)
#     # 0=relajado, 1=intermedio, 2=perturbado
